In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"], "figure.dpi": 110})

SLIM_DIR = "/Volumes/Lexar/mglurs-brain-sc-datasets/derived/regions"
FIG_DIR = "figures/allen_wmb"
RES_DIR = "results/allen_wmb"
NAMES = {'ACA': 'Anterior cingulate area', 'AI': 'Agranular insular area', 'AUD': 'Auditory areas', 'AUD-TEa-PERI-ECT': 'Auditory/temporal/perirhinal/ectorhinal areas', 'CB': 'Cerebellum', 'CTXsp': 'Cortical subplate', 'ENT': 'Entorhinal area', 'HIP': 'Hippocampal region', 'HY': 'Hypothalamus', 'LSX': 'Lateral septal complex', 'MB': 'Midbrain', 'MY': 'Medulla', 'OLF': 'Olfactory areas', 'P': 'Pons', 'PAL': 'Pallidum', 'RHP': 'Retrohippocampal region', 'RSP': 'Retrosplenial area', 'SSp': 'Primary somatosensory area', 'STRd': 'Striatum dorsal region', 'STRv': 'Striatum ventral region', 'TH': 'Thalamus', 'VIS': 'Visual areas', 'sAMY': 'Striatum-like amygdalar nuclei', 'MOp': 'Primary motor area', 'VIS-PTLp': 'Visual/posterior parietal  areas', 'TEa-PERI-ECT': 'Temporal/perirhinal/ectorhinal areas ', 'PL-ILA-ORB': 'Prelimbic/infralimbic/orbital areas ', 'SS-GU-VISC': 'Somatosensory/gustatory/visceral areas', 'MO-FRP': 'Somatomotor - Frontal pole'}
MIN_CELLS = 200
GRMS = [f"Grm{i}" for i in range(1, 9)]

_t20 = plt.get_cmap("tab20").colors
_all = [_t20[i] for i in range(0, 20, 2)] + [_t20[i] for i in range(1, 20, 2)] + list(plt.get_cmap("tab20b").colors)
PAL = [c for c in _all if not (abs(c[0] - c[1]) < 0.03 and abs(c[1] - c[2]) < 0.03)]

def class_num(c):
    n = pd.to_numeric(str(c)[:2], errors="coerce")
    return int(n) if pd.notna(n) else 0

def class_color(c):
    n = class_num(c)
    return PAL[(n - 1) % len(PAL)] if n > 0 else (0.5, 0.5, 0.5)

def is_neuron_class(c):
    return not (30 <= class_num(c) <= 34)

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

ROI = "HIP"
CHEMS = None

df = pd.read_parquet(f"{SLIM_DIR}/{ROI}_slim.parquet")
if CHEMS:
    df = df[df["chem"].isin(CHEMS)]
genes = [g for g in GRMS if g in df.columns]
tag = ROI + ("_" + "-".join(c.replace("WMB-10X", "") for c in CHEMS) if CHEMS else "")
rname = f"{NAMES.get(ROI, ROI)} ({ROI})"
chem_str = ", ".join(sorted(c.replace("WMB-10X", "10x ").replace("Multi", "Multiome") for c in df["chem"].astype(str).unique()))
print(ROI, len(df), "cells;", df["subclass"].nunique(), "subclasses; chem:", df["chem"].value_counts().to_dict())

d = df.dropna(subset=["allen_x", "allen_y"])
if len(d) > 150000:
    d = d.sample(150000, random_state=0)
classes = sorted(d["class"].unique())

def darken(c, f=0.55):
    return tuple(v * f for v in c[:3])

fig, ax = plt.subplots(figsize=(18, 13.5))
ax.scatter(d["allen_x"], d["allen_y"], s=1.5, c=[class_color(c) for c in d["class"]], linewidths=0, rasterized=True)
cent = d.groupby("subclass")[["allen_x", "allen_y"]].median()
cls_of = d.groupby("subclass")["class"].first()
xr, yr = np.ptp(d["allen_x"]), np.ptp(d["allen_y"])
placed = []
for sc in d["subclass"].value_counts().index[:120]:
    x, y = cent.loc[sc, "allen_x"], cent.loc[sc, "allen_y"]
    if any(abs(x - px) / xr < 0.2 and abs(y - py) / yr < 0.055 for px, py in placed):
        continue
    ax.text(x, y, sc, fontsize=18, color=darken(class_color(cls_of[sc])), ha="center", va="center", zorder=5,
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.65))
    placed.append((x, y))
    if len(placed) >= 35:
        break
handles = [plt.Line2D([], [], marker="o", ls="", color=class_color(c), label=c, markersize=14) for c in classes]
ax.legend(handles=handles, fontsize=18, title_fontsize=18, frameon=False, loc="center left", bbox_to_anchor=(1.0, 0.5), title="class")
ax.set_xlabel("Allen UMAP 1", fontsize=16)
ax.set_ylabel("Allen UMAP 2", fontsize=16)
ax.tick_params(labelsize=14)
ax.set_title(f"{rname}: {len(df):,} cells ({chem_str})\nAllen UMAP of single cells; color = class (legend); labels = largest subclasses that fit without overlap", fontsize=18)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
fig.savefig(f"{FIG_DIR}/{tag}_fig1_umap_subclass.png", dpi=200, bbox_inches="tight")
plt.show()